# Iris Flower Classification: A Beginner Machine Learning Project

## 1. Project idea
This project predicts the species of an Iris flower from four measurements:
- Sepal length (cm)
- Sepal width (cm)
- Petal length (cm)
- Petal width (cm)

The target is one of three species: **Setosa**, **Versicolor**, or **Virginica**. This can demonstrate how classification models learn patterns from labelled measurements.

## 2. Dataset
We use the Iris dataset bundled with scikit-learn. It contains 150 samples, four numerical features, and three classes.

Dataset documentation: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html

## 3. Libraries
We use pandas and NumPy for data handling, Matplotlib and Seaborn for visualisation, and scikit-learn for model training and evaluation.

## 4. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay

## 5. Load the dataset

In [ ]:
iris = load_iris(as_frame=True)

X = iris.data.copy()
y = iris.target.copy()

df = X.copy()
df["species"] = y.map(dict(enumerate(iris.target_names)))

print("Dataset shape:", df.shape)
display(df.head())

## 6. Data preparation and missing-value check
We check for missing values, separate the input features (`X`) from the target (`y`), and use a stratified train/test split. The test set is kept separate so it can evaluate how well the models perform on unseen samples.

In [ ]:
print("Missing values in each column:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())
print("\nClass distribution:")
print(df["species"].value_counts())

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))

## 7. Explore the data

In [ ]:
display(df.describe())

plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=df,
    x="petal length (cm)",
    y="petal width (cm)",
    hue="species"
)
plt.title("Petal length vs petal width by species")
plt.tight_layout()
plt.show()

## 8. Build two models

1. **Decision Tree:** learns a set of feature-based decision rules.
2. **K-Nearest Neighbors (KNN):** predicts a class based on the nearest training samples. Since KNN uses distances, we scale its features using `StandardScaler` inside a pipeline.

Both models use the same train/test split for a fair comparison.

In [ ]:
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42, max_depth=4),
    "KNN": make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=5)
    )
}

predictions = {}
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    predictions[name] = y_pred
    accuracy = accuracy_score(y_test, y_pred)
    results.append({"Model": name, "Accuracy": accuracy})

results_df = pd.DataFrame(results).sort_values("Accuracy", ascending=False)
display(results_df)

## 9. Evaluate the models
Accuracy is the fraction of test samples classified correctly. The confusion matrix compares actual species (rows) with predicted species (columns).

In [ ]:
for name, y_pred in predictions.items():
    print(f"--- {name} ---")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
    print(classification_report(
        y_test, y_pred,
        labels=[0, 1, 2],
        target_names=iris.target_names,
        zero_division=0
    ))

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=iris.target_names
    )
    disp.plot(cmap="Blues", values_format="d")
    plt.title(f"Confusion Matrix - {name}")
    plt.tight_layout()
    plt.show()

## 10. Compare model accuracy

In [ ]:
plt.figure(figsize=(7, 4))
sns.barplot(data=results_df, x="Model", y="Accuracy")
plt.ylim(0, 1.05)
plt.ylabel("Test accuracy")
plt.title("Decision Tree vs KNN")
plt.tight_layout()
plt.show()

best_row = results_df.iloc[0]
print(f"Best model on this test split: {best_row['Model']}")
print(f"Test accuracy: {best_row['Accuracy']:.3f}")

## 11. Try a prediction with new measurements
The example below uses a sample set of measurements in centimetres. It prints each model's predicted species. These are example inputs, not a guarantee of performance on every real-world flower.

In [ ]:
sample = pd.DataFrame([{
    "sepal length (cm)": 5.1,
    "sepal width (cm)": 3.5,
    "petal length (cm)": 1.4,
    "petal width (cm)": 0.2
}])

print("Input measurements:")
display(sample)

for name, model in models.items():
    predicted_class = model.predict(sample)[0]
    print(f"{name} prediction: {iris.target_names[predicted_class]}")

## 12. Conclusion

This project demonstrates a complete beginner classification workflow: loading a dataset, checking data quality, preparing features and target, training two models, and evaluating them with accuracy and confusion matrices.

After running all cells, use the actual accuracy scores and confusion matrices produced above to state which model performed better on this test split. Results may differ if the split or model settings change. A single test split is a useful first comparison, but it does not prove that one model will always be better.

## 13. What I learned
- How to separate features and target labels.
- Why a train/test split is needed.
- How a Decision Tree and KNN make predictions.
- Why feature scaling is useful for distance-based models such as KNN.
- How to use accuracy and a confusion matrix to assess classification.